In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import time
import copy
import jax
import jax.numpy as jnp
import viser

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

sys.path.append(os.path.abspath(".."))
from environments import RopeEnv
from planners import gpu_sls
from shapes import draw_half_ellipsoid, draw_cone

In [ ]:
server = viser.ViserServer()
_ = server.scene.add_grid(name="ground")

In [ ]:
env = RopeEnv(
    time_step=0.02,
    num_segments=15,
    rope_length=0.3,
    rope_diameter=0.004,
    youngs_modulus=1e5,
    mass_density=300,
    num_floating_grippers=2,
    grip_stiffness=800,
    gripper_radius=0.01,
    contact_smoothing=3e-3,
)

In [ ]:
def make_control_and_obstacle_constraints(
    u_min: jnp.ndarray,
    u_max: jnp.ndarray,
    cone_centers_xy: jnp.ndarray,
    cone_radius: float,
    cone_z_top: float,
):
    def constraints(x, u, t):
        control_constraints = jnp.concatenate((u - u_max, u_min - u))

        rope_nodes, _, gripper_pos = env.unpack_state(x)
        left_pos, right_pos = gripper_pos[0], gripper_pos[1]
        effector_constraints = jnp.array([left_pos[0], -right_pos[0]])

        node_xy = rope_nodes[:, 0:2]
        node_z = rope_nodes[:, 2]

        obstacle_constraints = []
        for cone_center_xy in cone_centers_xy:
            radial_dist = jnp.linalg.norm(node_xy - cone_center_xy[None, :], axis=1)
            z_required = cone_z_top * jnp.maximum(1.0 - (radial_dist / cone_radius), 0.0)

            cone_constraints = z_required - node_z
            cone_constraints = jnp.where(
                radial_dist <= cone_radius,
                cone_constraints,
                -1.0,
            )
            obstacle_constraints.append(cone_constraints)

        obstacle_constraints = jnp.concatenate(obstacle_constraints)

        return jnp.concatenate((control_constraints, effector_constraints, obstacle_constraints))

    return constraints


def make_constant_disturbance(alpha: float):
    def disturbance(X: jnp.ndarray) -> jnp.ndarray:
        N, nx = X.shape
        E0 = alpha * jnp.eye(nx, dtype=X.dtype)
        return jnp.broadcast_to(E0, (N, nx, nx))

    return disturbance


# Initial state
l = jnp.arange(env.params.num_nodes) * env.params.segment_length
l -= jnp.mean(l)
nodes = jnp.stack((l, jnp.zeros_like(l), jnp.full_like(l, 0.0)), axis=1)

x_grip = jnp.array([
    [-0.1, 0.0, 0.05],
    [ 0.1, 0.0, 0.05],
])
state0 = env.state(x_node=nodes, x_grip=x_grip)

# Target rope shape
state_goals = []

x_coords = jnp.arange(env.params.num_nodes) * env.params.segment_length
nodes = nodes.at[:, 2].set(0.05)
state_goals.append(env.state(x_node=nodes, x_grip=x_grip))

nodes = nodes.at[:, 1].set(0.5)
nodes = nodes.at[:, 2].set(0.1)
state_goals.append(env.state(x_node=nodes))

# Initial control (zero velocity)
control0 = env.control(c_grip=jnp.array([1.0] * env.params.num_floating_grippers))


N = 20  # Planning horizon
dt = env.params.dt

def make_cost(state_goal):
    def cost(W, reference, x, u, t):
        x_ref = state_goal
        u_ref = control0
        state_err = x - x_ref
        control_err = u - u_ref
        return (
            1.0 * jnp.sum(state_err[:-6]**2)
            + 0.1 * jnp.sum(control_err[:-2]**2)
        )

    return cost

def dynamics(x, u, t, parameter):
    return env.step(x, u)


vmax = 0.2
u_max = jnp.array([vmax, vmax, vmax, 10.0])
u_max = jnp.repeat(u_max, env.params.num_floating_grippers)

cone_centers_xy = jnp.array([[0.0, 0.25]])
cone_radius = 0.1
cone_z_max = 0.2

constraints = make_control_and_obstacle_constraints(
    u_min=-u_max,
    u_max=u_max,
    cone_centers_xy=cone_centers_xy,
    cone_radius=cone_radius * 1.1,
    cone_z_top=cone_z_max * 1.1,
)

draw_cone(
    server,
    name="/obstacles/cone",
    center_xy=cone_centers_xy[0],
    radius=cone_radius,
    z_min=0.0,
    z_max=cone_z_max,
)

admm_cfg = gpu_sls.ADMMConfig(
    eps_abs=5e-2,
    eps_rel=1e-2,
    rho_max=1e3,
    max_iterations=100,
    rho_update_frequency=25,
    initial_rho=10.0,
)

sls_cfg = gpu_sls.SLSConfig(
    max_sls_iterations=2,
    sls_primal_tol=1e-2,
    enable_fastsls=False,
    initialize_nominal=True,
    max_initial_sqp_iterations=0,
    warm_start=False,
    rti=False,
)

sqp_cfg = gpu_sls.SQPConfig(
    max_sqp_iterations=1,
    warm_start=False,
    feas_tol=1e-2,
    step_tol=1e-4,
    line_search=True,
)

nx = state0.size
cfg = gpu_sls.MPCConfig(
    n=nx,
    nu=control0.size,
    N=N,
    dt=dt,
    W=None,
    u_ref=control0,
)

obstacles = jnp.zeros((0, 3))
disturbance = make_constant_disturbance(alpha=0.003 * dt)
nc = constraints(state0, control0, 0.0).size

mpc_controllers = [None] * 2

X_in = jnp.tile(state0[None, :], (N + 1, 1))
U_in = jnp.tile(control0[None, :], (N, 1))
U_in = U_in.at[:, [2, 5]].set(-0.2)
for i in range(U_in.shape[0]):
    X_in = X_in.at[i+1].set(env.step(X_in[i], U_in[i]))
mpc_controllers[0] = gpu_sls.GenericMPC(
    sls_cfg,
    sqp_cfg,
    admm_cfg,
    config=cfg,
    dynamics=dynamics,
    cost=make_cost(state_goals[0]),
    constraints=constraints,
    obstacles=obstacles,
    disturbance=disturbance,
    num_constraints=nc,
    shift=1,
    X_in=X_in,
    U_in=U_in,
)

X_in = jnp.tile(state_goals[0][None, :], (N + 1, 1))
U_in = jnp.tile(control0[None, :], (N, 1))
mpc_controllers[1] = gpu_sls.GenericMPC(
    sls_cfg,
    sqp_cfg,
    admm_cfg,
    config=cfg,
    dynamics=dynamics,
    cost=make_cost(state_goals[1]),
    constraints=constraints,
    obstacles=obstacles,
    disturbance=disturbance,
    num_constraints=nc,
    shift=1,
    X_in=X_in,
    U_in=U_in,
)

In [ ]:
controller = copy.deepcopy(mpc_controllers[0])

state = state0
env.visualize(server, state)

for i in range(500):
    if i == 100:
        controller = copy.deepcopy(mpc_controllers[1])

    start = time.time()
    u0, X_pred, U_pred, V_pred, backoffs, Phi_x, Phi_u, Phi_xw, Phi_uw, Phi_xe, Phi_ue = controller.run(x0=state, reference=None, parameter=None, Xi=jnp.zeros((nx,nx)))
    elapsed = time.time() - start
    print(f"MPC step took {elapsed * 1e3:.2f} ms")

    control = jnp.clip(u0 if not jnp.isnan(u0).any() else control0, -u_max, u_max)
    state = env.step(state, control)

    if jnp.isnan(state).any():
        raise RuntimeError("NaN occurred in rope state")
    env.visualize(server, state)

    wait = dt - elapsed
    if wait > 0:
        time.sleep(wait)